# Clean Data

Apply heuristic text cleaning to both **English corpus** and **additional news data**.

Three-stage filter (see `clean_data_helper.py`):
1. **Symbol ratio** — rejects garbled OCR (non-word chars > 25% of words)
2. **Punctuation density** — rejects tables/lists (<12% of lines end with `.?!`)
3. **Stopwords** — rejects non-English text (fewer than 3 common stopwords)

Outputs cleaning masks (parquet files with `original_index` of clean rows):
- English: `D:\hist_LLM\corpus\cleaning_masks\{year}\{subset}_mask.parquet`
- Additional: `D:\hist_LLM\additional_data\cleaning_masks\{collection}\{year}_mask.parquet`

In [1]:
import os
import json
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import tempfile
import clean_data_helper

# ========================== CONFIG ==========================

# --- English corpus ---
ENGLISH_RAW_DIR = Path(r"D:\hist_LLM\corpus\raw")
ENGLISH_MASKS_DIR = Path(r"D:\hist_LLM\corpus\cleaning_masks")
ENGLISH_STATS_PATH = Path(r"D:\hist_LLM\corpus\cleaning_stats.csv")

# --- Additional news data ---
ADDITIONAL_RAW_DIR = Path(r"D:\hist_LLM\additional_data\raw")
ADDITIONAL_MASKS_DIR = Path(r"D:\hist_LLM\additional_data\cleaning_masks")
ADDITIONAL_STATS_PATH = Path(r"D:\hist_LLM\additional_data\cleaning_stats.csv")

ADDITIONAL_COLLECTIONS = {
    "nyt": {
        "raw_dir": ADDITIONAL_RAW_DIR / "news_archives" / "NYT_filtered_500char",
        "glob": "nyt_*.parquet",
        "text_col": "combined_text",
    },
    "economist": {
        "raw_dir": ADDITIONAL_RAW_DIR / "news_archives" / "Economist",
        "glob": "economist_*-*.parquet",
        "text_col": "ocr_text",
    },
    "ft": {
        "raw_dir": ADDITIONAL_RAW_DIR / "news_archives" / "FT",
        "glob": "*.parquet",
        "text_col": "text_cleaned",
    },
    "newswire": {
        "raw_dir": ADDITIONAL_RAW_DIR / "newswire",
        "glob": "*_data_clean.json",
        "text_col": "cleaned_article",
    },
}

# --- Processing settings ---
NUM_WORKERS = 8
START_YEAR = 1678
END_YEAR = 2023


def convert_newswire_to_parquet(json_dir: Path, temp_dir: Path) -> list:
    """Convert newswire JSON files to temp parquets for uniform processing.

    Names output files as {year}.parquet so masks become {year}_mask.parquet,
    matching the convention expected by downstream notebooks.
    """
    json_files = sorted(json_dir.glob("*_data_clean.json"))
    parquet_files = []
    for jf in tqdm(json_files, desc="Converting newswire JSON → parquet"):
        year = int(jf.name.split("_")[0])
        if year == 1957:  # Known corrupted file
            continue
        try:
            with open(jf, "r", encoding="utf-8") as f:
                data = json.load(f)
            texts = [doc.get("cleaned_article", "") for doc in data]
            df = pd.DataFrame({"cleaned_article": texts})
            pq_path = temp_dir / f"{year}.parquet"
            df.to_parquet(pq_path, index=False)
            parquet_files.append(pq_path)
        except Exception as e:
            print(f"  [!] {jf.name}: {e}")
    return parquet_files

## 1. Clean English Corpus

Parallel processing across all `subset_*.parquet` files for each year.
Masks saved to `D:\hist_LLM\corpus\cleaning_masks\{year}\`

In [2]:
target_years = range(START_YEAR, END_YEAR + 1)
all_files = [f for y in target_years for f in (ENGLISH_RAW_DIR / str(y)).glob("subset_*.parquet")]
print(f"Found {len(all_files)} English parquet files across {len(list(target_years))} years")

# Delete old stats file to avoid appending to stale data
if ENGLISH_STATS_PATH.exists():
    ENGLISH_STATS_PATH.unlink()

results = []
with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {
        executor.submit(
            clean_data_helper.process_parquet_file,
            f, str(ENGLISH_MASKS_DIR / f.parent.name), "text"
        ): f for f in all_files
    }
    for future in tqdm(as_completed(futures), total=len(all_files), desc="English"):
        res = future.result()
        results.append(res)
        if "error" in res:
            print(f"\n  [!] FAILED: {res.get('file')} | {res.get('error')}")

# Save stats
stats_df = pd.DataFrame([r for r in results if "error" not in r])
stats_df.to_csv(ENGLISH_STATS_PATH, index=False)

total = stats_df["total_rows"].sum()
clean = stats_df["clean_rows"].sum()
print(f"\nEnglish corpus: {clean:,} / {total:,} clean ({100*clean/total:.1f}%)")
print(f"Stats saved to: {ENGLISH_STATS_PATH}")

Found 34598 English parquet files across 346 years


English:   0%|          | 0/34598 [00:00<?, ?it/s]


English corpus: 65,007,337 / 125,520,261 clean (51.8%)
Stats saved to: D:\hist_LLM\corpus\cleaning_stats.csv


## 2. Clean Additional News Data

Process each collection with the same `process_parquet_file` pipeline.
Newswire JSONs are converted to temp parquets first for uniform handling.

Masks saved to `D:\hist_LLM\additional_data\cleaning_masks\{collection}\`

In [ ]:
all_additional_stats = []

for coll_name, cfg in ADDITIONAL_COLLECTIONS.items():
    print(f"\n{'='*50}")
    print(f"Collection: {coll_name}")
    print(f"{'='*50}")

    coll_masks_dir = ADDITIONAL_MASKS_DIR / coll_name
    text_col = cfg["text_col"]

    # Get parquet files — convert newswire JSON first
    if coll_name == "newswire":
        temp_dir = Path(tempfile.mkdtemp())
        all_files = convert_newswire_to_parquet(cfg["raw_dir"], temp_dir)
    else:
        all_files = sorted(cfg["raw_dir"].glob(cfg["glob"]))

    print(f"Found {len(all_files)} files to process")

    # Same parallel pipeline for all collections
    results = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {
            executor.submit(
                clean_data_helper.process_parquet_file,
                f, str(coll_masks_dir), text_col
            ): f for f in all_files
        }
        for future in tqdm(as_completed(futures), total=len(all_files), desc=coll_name):
            res = future.result()
            results.append({"collection": coll_name, **res})
            if "error" in res:
                print(f"  [!] {res.get('file')}: {res.get('error')}")

    # Clean up temp files for newswire
    if coll_name == "newswire":
        for f in temp_dir.glob("*.parquet"):
            f.unlink()
        temp_dir.rmdir()

    all_additional_stats.extend(results)

    # Print collection summary
    coll_stats = [s for s in results if "error" not in s]
    if coll_stats:
        coll_df = pd.DataFrame(coll_stats)
        total = coll_df["total_rows"].sum()
        clean = coll_df["clean_rows"].sum()
        print(f"  {coll_name}: {clean:,} / {total:,} clean ({100*clean/total:.1f}%)")

# Save all additional stats
add_stats_df = pd.DataFrame([s for s in all_additional_stats if "error" not in s])
add_stats_df.to_csv(ADDITIONAL_STATS_PATH, index=False)
print(f"\nAdditional stats saved to: {ADDITIONAL_STATS_PATH}")

## 3. Summary

In [ ]:
print("=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

# English
if ENGLISH_STATS_PATH.exists():
    eng = pd.read_csv(ENGLISH_STATS_PATH)
    eng_total = eng["total_rows"].sum()
    eng_clean = eng["clean_rows"].sum()
    print(f"\nEnglish corpus:")
    print(f"  Total:   {eng_total:>12,}")
    print(f"  Clean:   {eng_clean:>12,} ({100*eng_clean/eng_total:.1f}%)")
    print(f"  Removed: {eng_total - eng_clean:>12,} ({100*(eng_total-eng_clean)/eng_total:.1f}%)")

# Additional
if ADDITIONAL_STATS_PATH.exists():
    add = pd.read_csv(ADDITIONAL_STATS_PATH)
    print(f"\nAdditional collections:")
    for coll in add["collection"].unique():
        coll_df = add[add["collection"] == coll]
        t = coll_df["total_rows"].sum()
        c = coll_df["clean_rows"].sum()
        print(f"  {coll:12s}: {c:>10,} / {t:>10,} clean ({100*c/t:.1f}%)")
    add_total = add["total_rows"].sum()
    add_clean = add["clean_rows"].sum()
    print(f"  {'TOTAL':12s}: {add_clean:>10,} / {add_total:>10,} clean ({100*add_clean/add_total:.1f}%)")

# Combined
if ENGLISH_STATS_PATH.exists() and ADDITIONAL_STATS_PATH.exists():
    grand_total = eng_total + add_total
    grand_clean = eng_clean + add_clean
    print(f"\nCombined: {grand_clean:,} / {grand_total:,} clean ({100*grand_clean/grand_total:.1f}%)")